# MICEX API Analysis - Options and Securities

This notebook demonstrates how to work with the Moscow Exchange (MICEX) API for:
- Getting underlying assets for options
- Analyzing options data
- Fetching securities and market data

API Reference: https://iss.moex.com/iss/reference/


In [2]:
# Setup and imports
import requests
import pandas as pd
import json
from datetime import date, datetime, timedelta
# import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Base MICEX API URL
BASE_URL = "https://iss.moex.com/iss"

print("MICEX API Analysis Setup Complete")
print(f"Base URL: {BASE_URL}")


MICEX API Analysis Setup Complete
Base URL: https://iss.moex.com/iss


## 1. Options Assets - Underlying Securities

Using the endpoint: `/iss/statistics/engines/futures/markets/options/assets`

This endpoint provides option volumes aggregated by underlying asset.

**Parameters from API documentation:**
- `date`: Date (default: today)
- `asset_type`: Filter by asset type: 'S' - Options on stocks
- `margin_style`: Margin style: 'M' - margined, 'P' - premium  
- `option_on_spot`: 0 - futures, 1 - spot instrument
- `limit`: Limit rows (10, 20, 100, 200, 1000, 10000)


In [4]:
def get_options_underlying_assets(date_param=None, asset_type=None, margin_style=None, option_on_spot=None, limit=None):
    """
    Get underlying assets for options from MICEX API
    
    Parameters:
    - date_param: Date (default: today)
    - asset_type: Filter by asset type: 'S' - Options on stocks
    - margin_style: Margin style: 'M' - margined, 'P' - premium
    - option_on_spot: 0 - futures, 1 - spot instrument
    - limit: Limit number of rows (10, 20, 100, 200, 1000, 10000)
    """
    
    url = f"{BASE_URL}/statistics/engines/futures/markets/options/assets.json"
    
    # Build parameters
    params = {}
    if date_param:
        params['date'] = date_param
    if asset_type:
        params['asset_type'] = asset_type
    if margin_style:
        params['margin_style'] = margin_style
    if option_on_spot is not None:
        params['option_on_spot'] = option_on_spot
    if limit:
        params['limit'] = limit
    
    print(f"Requesting: {url}")
    print(f"Parameters: {params}")
    
    try:
        response = requests.get(url, params=params)
        print(f"Response status: {response.status_code}")
        
        if response.status_code == 200:
            data = response.json()
            
            print(f"Available sections: {list(data.keys())}")
            
            result = {}
            for section_name, section_data in data.items():
                if isinstance(section_data, dict) and 'data' in section_data and 'columns' in section_data:
                    if section_data['data']:
                        df = pd.DataFrame(section_data['data'], columns=section_data['columns'])
                        result[section_name] = df
                        print(f"Section '{section_name}': {len(df)} rows, columns: {list(df.columns)}")
                    else:
                        print(f"Section '{section_name}': No data")
            
            return result
        else:
            print(f"Error: HTTP {response.status_code}")
            print(f"Response: {response.text}")
            return None
            
    except Exception as e:
        print(f"Error fetching options assets: {e}")
        return None

# Get all underlying assets for options
print("Fetching underlying assets for options...")
options_assets = get_options_underlying_assets(asset_type='S', margin_style='P', option_on_spot=1)

if options_assets:
    for section_name, df in options_assets.items():
        print(f"\n{'='*50}")
        print(f"SECTION: {section_name.upper()}")
        print(f"{'='*50}")
        print(df.head(10))
        
        if len(df) > 10:
            print(f"\n... and {len(df) - 10} more rows")


Fetching underlying assets for options...
Requesting: https://iss.moex.com/iss/statistics/engines/futures/markets/options/assets.json
Parameters: {'asset_type': 'S', 'margin_style': 'P', 'option_on_spot': 1}
Response status: 200
Available sections: ['asset_volumes']
Section 'asset_volumes': 41 rows, columns: ['tradedate', 'asset', 'shortname', 'asset_type', 'asset_last_price', 'asset_last_to_prev_price', 'asset_high_price', 'asset_low_price', 'valtoday', 'voltoday', 'numtrades', 'openposition', 'oichange', 'updatetime', 'option_secid', 'margin_style', 'option_on_spot']

SECTION: ASSET_VOLUMES
    tradedate asset   shortname asset_type  asset_last_price  \
0  2025-06-19  SBER    Сбербанк          S            313.68   
1  2025-06-19  GAZP  ГАЗПРОМ ао          S            126.18   
2  2025-06-19  MOEX    МосБиржа          S            189.70   
3  2025-06-19  MAGN         ММК          S             31.77   
4  2025-06-19  ALRS   АЛРОСА ао          S             44.97   
5  2025-06-19  R

In [5]:
# Filter for stock options specifically
print("\nFetching stock options (asset_type='S')...")
stock_options = get_options_underlying_assets(asset_type='S', margin_style='P', option_on_spot=1, limit=100)

if stock_options:
    print("\nSTOCK OPTIONS ANALYSIS:")
    print("="*40)
    
    for section_name, df in stock_options.items():
        print(f"\n{section_name.upper()} section:")
        
        if not df.empty:
            print(f"Columns: {list(df.columns)}")
            print(f"Total records: {len(df)}")
            
            # Show sample data
            print("\nSample data:")
            print(df.head())
            
            # Look for specific columns that might contain underlying names
            name_columns = [col for col in df.columns if 'name' in col.lower() or 'asset' in col.lower() or 'secid' in col.lower()]
            if name_columns:
                print(f"\nUnderlying assets found in columns {name_columns}:")
                for col in name_columns:
                    unique_values = df[col].unique()[:10]  # Show first 10 unique values
                    print(f"  {col}: {list(unique_values)}")
                    
            # Look for Sberbank specifically
            sber_columns = [col for col in df.columns if df[col].dtype == 'object']
            sber_found = False
            for col in sber_columns:
                sber_mask = df[col].astype(str).str.contains('Сбербанк|SBER|сбер', case=False, na=False)
                if sber_mask.any():
                    sber_data = df[sber_mask]
                    print(f"\nFound Sberbank references in column '{col}':")
                    print(sber_data[[col] + [c for c in ['ASSETCODE', 'SHORTNAME'] if c in df.columns]][:5])
                    sber_found = True
            
            if not sber_found:
                print("\nNo Sberbank references found in this section")
        else:
            print("No data in this section")



Fetching stock options (asset_type='S')...
Requesting: https://iss.moex.com/iss/statistics/engines/futures/markets/options/assets.json
Parameters: {'asset_type': 'S', 'margin_style': 'P', 'option_on_spot': 1, 'limit': 100}
Response status: 200
Available sections: ['asset_volumes']
Section 'asset_volumes': 41 rows, columns: ['tradedate', 'asset', 'shortname', 'asset_type', 'asset_last_price', 'asset_last_to_prev_price', 'asset_high_price', 'asset_low_price', 'valtoday', 'voltoday', 'numtrades', 'openposition', 'oichange', 'updatetime', 'option_secid', 'margin_style', 'option_on_spot']

STOCK OPTIONS ANALYSIS:

ASSET_VOLUMES section:
Columns: ['tradedate', 'asset', 'shortname', 'asset_type', 'asset_last_price', 'asset_last_to_prev_price', 'asset_high_price', 'asset_low_price', 'valtoday', 'voltoday', 'numtrades', 'openposition', 'oichange', 'updatetime', 'option_secid', 'margin_style', 'option_on_spot']
Total records: 41

Sample data:
    tradedate asset   shortname asset_type  asset_la

## 2. Get Specific Asset and Expirations

Now let's get expiration dates for a specific asset (e.g., Sberbank) and then get the actual options data.


In [10]:
def get_asset_expirations(asset_code):
    """
    Get expiration dates for a specific asset
    Endpoint: /iss/statistics/engines/futures/markets/options/assets/[asset]
    """
    url = f"{BASE_URL}/statistics/engines/futures/markets/options/assets/{asset_code}.json"
    
    print(f"Getting expirations for asset: {asset_code}")
    print(f"URL: {url}")
    
    try:
        response = requests.get(url)
        print(f"Response status: {response.status_code}")
        
        if response.status_code == 200:
            data = response.json()
            print(f"Available sections: {list(data.keys())}")
            
            result = {}
            for section_name, section_data in data.items():
                if isinstance(section_data, dict) and 'data' in section_data and 'columns' in section_data:
                    if section_data['data']:
                        df = pd.DataFrame(section_data['data'], columns=section_data['columns'])
                        result[section_name] = df
                        print(f"Section '{section_name}': {len(df)} rows, columns: {list(df.columns)}")
                    else:
                        print(f"Section '{section_name}': No data")
            
            return result
        else:
            print(f"Error: HTTP {response.status_code}")
            print(f"Response: {response.text}")
            return None
            
    except Exception as e:
        print(f"Error fetching asset expirations: {e}")
        return None

# First, let's see what assets we have from the previous cell
if 'stock_options' in locals() and stock_options:
    print("Looking for asset codes in the underlying assets data...")
    
    # Find asset codes from the first call
    for section_name, df in stock_options.items():
        if not df.empty:
            print(f"\nSection: {section_name}")
            print(f"Columns: {list(df.columns)}")
            
            # Look for asset-related columns
            asset_columns = [col for col in df.columns if 'asset' in col.lower() or 'code' in col.lower() or 'secid' in col.lower()]
            if asset_columns:
                print(f"Asset-related columns: {asset_columns}")
                for col in asset_columns:
                    unique_values = df[col].unique()[:5]
                    print(f"  {col} samples: {list(unique_values)}")
                    
                    # Try to find Sberbank-related asset
                    sber_mask = df[col].astype(str).str.contains('SBER|сбер', case=False, na=False)
                    if sber_mask.any():
                        sber_assets = df[sber_mask][col].unique()
                        print(f"  Found Sberbank assets: {list(sber_assets)}")
                        
                        # Try the first Sberbank asset
                        if len(sber_assets) > 0:
                            working_asset = sber_assets[0]
                            print(f"\nTesting asset: {working_asset}")
                            asset_data = get_asset_expirations(working_asset)
                            
                            if asset_data:
                                print(f"✓ Successfully got expiration data for {working_asset}")
                                break
else:
    # Try some common asset codes for Sberbank
    test_assets = ['SBER', 'SBERP', 'SBPR']
    
    print("Testing common Sberbank asset codes...")
    for asset_code in test_assets:
        print(f"\nTrying asset code: {asset_code}")
        asset_data = get_asset_expirations(asset_code)
        
        if asset_data:
            print(f"✓ Found data for asset: {asset_code}")
            working_asset = asset_code
            break
    else:
        asset_data = None
        working_asset = None
        print("No working asset codes found")


Looking for asset codes in the underlying assets data...

Section: asset_volumes
Columns: ['tradedate', 'asset', 'shortname', 'asset_type', 'asset_last_price', 'asset_last_to_prev_price', 'asset_high_price', 'asset_low_price', 'valtoday', 'voltoday', 'numtrades', 'openposition', 'oichange', 'updatetime', 'option_secid', 'margin_style', 'option_on_spot']
Asset-related columns: ['asset', 'asset_type', 'asset_last_price', 'asset_last_to_prev_price', 'asset_high_price', 'asset_low_price', 'option_secid']
  asset samples: ['SBER', 'GAZP', 'MOEX', 'MAGN', 'ALRS']
  Found Sberbank assets: ['SBER', 'SBERP']

Testing asset: SBER
Getting expirations for asset: SBER
URL: https://iss.moex.com/iss/statistics/engines/futures/markets/options/assets/SBER.json
Response status: 200
Available sections: ['expirations']
Section 'expirations': 14 rows, columns: ['series_type', 'expiration_date']
✓ Successfully got expiration data for SBER


## 3. Get Option Board Data (CALL and PUT Options)

Using the endpoint: `/iss/statistics/engines/futures/markets/options/assets/[asset]/optionboard`

This gives us the actual CALL and PUT options for specific expiration dates.


In [15]:
def get_option_board(asset_code, expiration_date=None):
    """
    Get option board data (CALL and PUT options) for an asset
    Endpoint: /iss/statistics/engines/futures/markets/options/assets/[asset]/optionboard
    API Reference: https://iss.moex.com/iss/reference/9
    
    Parameters:
    - asset_code: The asset code (e.g., 'SBER')
    - expiration_date: Expiration date in YYYY-MM-DD format (optional)
    """
    url = f"{BASE_URL}/statistics/engines/futures/markets/options/assets/{asset_code}/optionboard.json"
    
    params = {}
    if expiration_date:
        params['date'] = expiration_date
    
    print(f"Getting option board for asset: {asset_code}")
    print(f"URL: {url}")
    print(f"Parameters: {params}")
    
    try:
        response = requests.get(url, params=params)
        print(f"Response status: {response.status_code}")
        
        if response.status_code == 200:
            data = response.json()
            print(f"Available sections: {list(data.keys())}")
            
            result = {}
            for section_name, section_data in data.items():
                if isinstance(section_data, dict) and 'data' in section_data and 'columns' in section_data:
                    if section_data['data']:
                        df = pd.DataFrame(section_data['data'], columns=section_data['columns'])
                        result[section_name] = df
                        print(f"Section '{section_name}': {len(df)} rows")
                        print(f"  Columns: {list(df.columns)}")
                        
                        # Show sample data
                        if len(df) > 0:
                            print(f"  Sample data:")
                            display_cols = df.columns[:8] if len(df.columns) > 8 else df.columns
                            print(df[display_cols].head(3).to_string())
                    else:
                        print(f"Section '{section_name}': No data")
            
            return result
        else:
            print(f"Error: HTTP {response.status_code}")
            print(f"Response: {response.text}")
            return None
            
    except Exception as e:
        print(f"Error fetching option board: {e}")
        return None

# Get option board data if we have a working asset
if 'working_asset' in locals() and working_asset:
    print(f"Getting option board for asset: {working_asset}")
    option_board_data = get_option_board(working_asset)
    
    if option_board_data:
        print(f"\n{'='*60}")
        print(f"OPTION BOARD DATA FOR {working_asset}")
        print(f"{'='*60}")
        
        # Analyze the option board data
        for section_name, df in option_board_data.items():
            if not df.empty:
                print(f"\nSection: {section_name}")
                print(f"Total options: {len(df)}")
                
                # Look for CALL/PUT indicators
                if 'OPTIONTYPE' in df.columns:
                    option_types = df['OPTIONTYPE'].value_counts()
                    print(f"Option types: {dict(option_types)}")
                
                # Look for strikes
                if 'STRIKE' in df.columns:
                    strikes = sorted(df['STRIKE'].unique())[:10]
                    print(f"Sample strikes: {strikes}")
                
                # Look for expiration dates
                exp_columns = [col for col in df.columns if 'exp' in col.lower() or 'date' in col.lower()]
                for col in exp_columns:
                    unique_dates = df[col].unique()[:5]
                    print(f"Expiration dates in {col}: {list(unique_dates)}")
    
    # If we have expiration data, try with specific expiration
    if 'asset_data' in locals() and asset_data:
        print(f"\nTrying with specific expiration dates...")
        
        # Look for expiration dates in the asset data
        for section_name, df in asset_data.items():
            if not df.empty:
                exp_columns = [col for col in df.columns if 'exp' in col.lower() or 'date' in col.lower()]
                for col in exp_columns:
                    exp_dates = df[col].unique()[1:3]  # Try first 3 expiration dates
                    
                    for exp_date in exp_dates:
                        if pd.notna(exp_date):
                            print(f"\nTrying with expiration: {exp_date}")
                            specific_board = get_option_board(working_asset, str(exp_date))
                            
                            if specific_board:
                                print(f"✓ Got data for expiration {exp_date}")
                                option_board_specific = specific_board
                                break
                    break
                break

else:
    print("No working asset found. Try running the previous cell first.")
    option_board_data = None


Getting option board for asset: SBER
Getting option board for asset: SBER
URL: https://iss.moex.com/iss/statistics/engines/futures/markets/options/assets/SBER/optionboard.json
Parameters: {}
Response status: 200
Available sections: ['call', 'put', 'asset']
Section 'call': 12 rows
  Columns: ['SECID', 'BOARDID', 'STRIKE', 'THEORPRICE', 'VOLAT', 'LAST', 'BID', 'OFFER', 'VOLTODAY', 'OPENPOSITION']
  Sample data:
       SECID BOARDID  STRIKE  THEORPRICE     VOLAT  LAST   BID  OFFER
0  SR260CF5D    ROPD   260.0       53.43  65.05791   0.0  45.3   0.00
1  SR270CF5D    ROPD   270.0       43.47  54.29826   0.0  35.9   0.00
2  SR280CF5D    ROPD   280.0       33.52  43.34361   0.0  32.7  35.16
Section 'put': 12 rows
  Columns: ['SECID', 'BOARDID', 'STRIKE', 'THEORPRICE', 'VOLAT', 'LAST', 'BID', 'OFFER', 'VOLTODAY', 'OPENPOSITION']
  Sample data:
       SECID BOARDID  STRIKE  THEORPRICE     VOLAT  LAST   BID  OFFER
0  SR260CR5D    ROPD   260.0        0.10  65.05791   0.0  0.00   0.89
1  SR270CR5D

## 4. Get Additional Option Data

Using additional endpoints for more detailed option information:
- **Reference 145**: `/iss/engines/futures/markets/options/boards/[board]/securities/[security]`
- **Reference 49**: `/iss/engines/futures/markets/options/boards/[board]/securities/[security]/orderbook`  
- **Reference 119**: `/iss/engines/futures/markets/options/boards/[board]/securities/[security]/trades`


In [17]:
def get_option_detailed_data(option_secid, asset_code, board="ROPD"):
    """
    Get detailed data for a specific option contract and asset
    
    Parameters:
    - option_secid: The option security ID
    - asset_code: The underlying asset code (e.g., 'SBER')
    - board: The trading board (default: ROPD)
    """
    
    # Different endpoints for detailed option data
    endpoints = {
        'security_info': f"/engines/futures/markets/options/boards/{board}/securities/{option_secid}.json",
        'trades': f"/engines/futures/markets/options/boards/{board}/securities/{option_secid}/trades.json",
        'turnovers': f"/statistics/engines/futures/markets/options/assets/{asset_code}/turnovers.json",
        'open_positions': f"/statistics/engines/futures/markets/options/assets/{asset_code}/openpositions.json"
    }
    
    all_data = {}
    
    print(f"Getting detailed data for option: {option_secid}")
    print(f"Asset: {asset_code}, Board: {board}")
    
    for data_type, endpoint in endpoints.items():
        url = f"{BASE_URL}{endpoint}"
        
        print(f"\n{'='*50}")
        print(f"FETCHING: {data_type.upper()}")
        print(f"Endpoint: {endpoint}")
        print(f"{'='*50}")
        
        try:
            response = requests.get(url)
            print(f"Status: {response.status_code}")
            
            if response.status_code == 200:
                data = response.json()
                print(f"Sections: {list(data.keys())}")
                
                section_data = {}
                for section_name, section_info in data.items():
                    if isinstance(section_info, dict) and 'data' in section_info and 'columns' in section_info:
                        if section_info['data']:
                            df = pd.DataFrame(section_info['data'], columns=section_info['columns'])
                            section_data[section_name] = df
                            print(f"  {section_name}: {len(df)} rows, {len(df.columns)} columns")
                            
                            # Show data
                            if len(df) > 0:
                                print(f"    Columns: {list(df.columns)}")
                                print(f"    Sample data:")
                                display_cols = df.columns[:6] if len(df.columns) > 6 else df.columns
                                print(df[display_cols].head(2).to_string())
                        else:
                            print(f"  {section_name}: No data")
                
                if section_data:
                    all_data[data_type] = section_data
                    print(f"✓ Successfully retrieved {data_type} data")
                
            elif response.status_code == 404:
                print(f"  Endpoint not found (404)")
            else:
                print(f"  Error: HTTP {response.status_code}")
                print(f"  Response: {response.text[:200]}...")
                
        except Exception as e:
            print(f"  Error: {e}")
    
    return all_data

def get_volume_and_positions_data(asset_code, expiration_date=None):
    """
    Get volume and open positions data for an asset
    API References:
    - Turnovers: https://iss.moex.com/iss/statistics/engines/futures/markets/options/assets/SBER/turnovers
    - Open Positions: https://iss.moex.com/iss/statistics/engines/futures/markets/options/assets/SBER/openpositions
    
    Parameters:
    - asset_code: The underlying asset code (e.g., 'SBER')
    - expiration_date: Expiration date for open positions (optional)
    """
    
    endpoints = {
        'turnovers': f"/statistics/engines/futures/markets/options/assets/{asset_code}/turnovers.json",
        'open_positions': f"/statistics/engines/futures/markets/options/assets/{asset_code}/openpositions.json"
    }
    
    all_data = {}
    
    print(f"Getting volume and positions data for asset: {asset_code}")
    
    for data_type, endpoint in endpoints.items():
        url = f"{BASE_URL}{endpoint}"
        
        params = {}
        if data_type == 'open_positions' and expiration_date:
            params['date'] = expiration_date
        
        print(f"\n{'='*50}")
        print(f"FETCHING: {data_type.upper()}")
        print(f"URL: {url}")
        if params:
            print(f"Parameters: {params}")
        print(f"{'='*50}")
        
        try:
            response = requests.get(url, params=params)
            print(f"Status: {response.status_code}")
            
            if response.status_code == 200:
                data = response.json()
                print(f"Sections: {list(data.keys())}")
                
                section_data = {}
                for section_name, section_info in data.items():
                    if isinstance(section_info, dict) and 'data' in section_info and 'columns' in section_info:
                        if section_info['data']:
                            df = pd.DataFrame(section_info['data'], columns=section_info['columns'])
                            section_data[section_name] = df
                            print(f"  {section_name}: {len(df)} rows")
                            print(f"    Columns: {list(df.columns)}")
                            
                            # Show sample data
                            if len(df) > 0:
                                print(f"    Sample data:")
                                display_cols = df.columns[:8] if len(df.columns) > 8 else df.columns
                                print(df[display_cols].head(3).to_string())
                        else:
                            print(f"  {section_name}: No data")
                
                if section_data:
                    all_data[data_type] = section_data
                    print(f"✓ Successfully retrieved {data_type} data")
                
            elif response.status_code == 404:
                print(f"  Endpoint not found (404)")
            else:
                print(f"  Error: HTTP {response.status_code}")
                
        except Exception as e:
            print(f"  Error: {e}")
    
    return all_data

# Get volume and positions data first
if 'working_asset' in locals() and working_asset:
    print(f"Getting volume and positions data for {working_asset}...")
    volume_positions_data = get_volume_and_positions_data(working_asset)
    
    # Also try with specific expiration date if available
    if 'option_board_data' in locals() and option_board_data:
        for section_name, df in option_board_data.items():
            if section_name == 'asset' and 'LASTDELDATE' in df.columns:
                exp_date = df['LASTDELDATE'].iloc[0]
                print(f"\nTrying with expiration date: {exp_date}")
                volume_positions_exp = get_volume_and_positions_data(working_asset, exp_date)
                break

# Get detailed data for specific options if we have option board data
if 'option_board_data' in locals() and option_board_data and 'working_asset' in locals():
    print("\n" + "="*80)
    print("GETTING DETAILED OPTION DATA")
    print("="*80)
    
    # Find option SECIDs
    option_secids = []
    for section_name, df in option_board_data.items():
        if not df.empty and 'SECID' in df.columns:
            secids = df['SECID'].dropna().unique()[:2]  # Get first 2 options
            option_secids.extend(secids)
            print(f"Found SECIDs in {section_name}: {list(secids)}")
    
    if option_secids:
        # Get detailed data for the first option
        first_option = option_secids[0]
        print(f"\nAnalyzing option: {first_option}")
        
        detailed_option_data = get_option_detailed_data(first_option, working_asset)
        
        if detailed_option_data:
            print(f"\n{'='*60}")
            print("SUMMARY OF DETAILED OPTION DATA:")
            print(f"{'='*60}")
            
            for data_type, sections in detailed_option_data.items():
                print(f"\n{data_type.upper()}:")
                for section_name, df in sections.items():
                    print(f"  {section_name}: {len(df)} rows")
                    
                    # Show key information based on data type
                    if data_type == 'security_info' and len(df) > 0:
                        print("    Security Information:")
                        key_cols = [col for col in df.columns if col.upper() in ['SECNAME', 'STRIKE', 'OPTIONTYPE', 'EXPDATE', 'LASTPRICE']]
                        available_cols = [col for col in key_cols if col in df.columns]
                        if available_cols:
                            print(df[available_cols].head(1).to_string())
                    
                    elif data_type == 'trades' and len(df) > 0:
                        print("    Recent Trades:")
                        trade_cols = [col for col in df.columns if col.upper() in ['TIME', 'PRICE', 'QUANTITY', 'VALUE', 'TRADENO']]
                        available_trade_cols = [col for col in trade_cols if col in df.columns]
                        if available_trade_cols:
                            print(df[available_trade_cols].head(3).to_string())
                    
                    elif data_type == 'turnovers' and len(df) > 0:
                        print("    Volume Data:")
                        vol_cols = [col for col in df.columns if 'VOL' in col.upper() or 'TURNOVER' in col.upper() or 'VALUE' in col.upper()]
                        if vol_cols:
                            print(df[vol_cols[:6]].head(3).to_string())
                    
                    elif data_type == 'open_positions' and len(df) > 0:
                        print("    Open Positions:")
                        pos_cols = [col for col in df.columns if 'POSITION' in col.upper() or 'OPEN' in col.upper()]
                        if pos_cols:
                            print(df[pos_cols[:6]].head(3).to_string())
        
        # Store for further use
        selected_option_secid = first_option
        selected_option_data = detailed_option_data
        
    else:
        print("No option SECIDs found in board data")
        
else:
    print("No option board data available. Run previous cells first.")


Getting volume and positions data for SBER...
Getting volume and positions data for asset: SBER

FETCHING: TURNOVERS
URL: https://iss.moex.com/iss/statistics/engines/futures/markets/options/assets/SBER/turnovers.json
Status: 200
Sections: ['asset_turnovers']
  asset_turnovers: 30 rows
    Columns: ['asset', 'tradedate', 'voltoday', 'openposition']
    Sample data:
  asset   tradedate  voltoday  openposition
0  SBER  2025-06-20     10472       5787790
1  SBER  2025-06-19    260657       5770822
2  SBER  2025-06-18    223563       8434530
✓ Successfully retrieved turnovers data

FETCHING: OPEN_POSITIONS
URL: https://iss.moex.com/iss/statistics/engines/futures/markets/options/assets/SBER/openpositions.json
Status: 200
Sections: ['open_positions']
  open_positions: 2 rows
    Columns: ['tradedate', 'asset', 'option_asset_type', 'option_type', 'is_fiz', 'persons_long', 'persons_short', 'open_position_long', 'open_position_short', 'oichange_long', 'oichange_short']
    Sample data:
    trade

## 5. Summary - Complete Options Data Flow

This shows the complete workflow for getting options data from MICEX API.


In [18]:
# Updated summary with corrections
print("MICEX OPTIONS API - CORRECTED WORKFLOW")
print("="*80)

print("\n📋 UPDATED STEP-BY-STEP PROCESS:")
print("="*40)

print("\n1️⃣ GET UNDERLYING ASSETS:")
print("   Endpoint: /iss/statistics/engines/futures/markets/options/assets.json")
print("   Purpose: Find which assets have options available")
print("   Parameters: asset_type='S' for stocks")

print("\n2️⃣ GET ASSET EXPIRATIONS:")
print("   Endpoint: /iss/statistics/engines/futures/markets/options/assets/[ASSET].json")
print("   Purpose: Get expiration dates for specific asset")
print("   Example: /iss/statistics/engines/futures/markets/options/assets/SBER.json")

print("\n3️⃣ GET OPTION BOARD:")
print("   Endpoint: /iss/statistics/engines/futures/markets/options/assets/[ASSET]/optionboard.json")
print("   Purpose: Get all CALL and PUT options for the asset")
print("   Parameters: date=[expiration_date] (optional)")

print("\n4️⃣ GET DETAILED OPTION DATA:")
print("   🔹 Security Info: /iss/engines/futures/markets/options/boards/ROPD/securities/[SECID].json")
print("   🔹 Trades: /iss/engines/futures/markets/options/boards/ROPD/securities/[SECID]/trades.json") 
print("   🔹 Volume Data: /iss/statistics/engines/futures/markets/options/assets/[ASSET]/turnovers.json")
print("   🔹 Open Positions: /iss/statistics/engines/futures/markets/options/assets/[ASSET]/openpositions.json")
print("   ⚠️  Note: Order book requires subscription, ROPD is the correct board")

print("\n" + "="*80)
print("🔧 KEY CORRECTIONS MADE:")
print("="*80)
print("✅ Changed board from 'OPEU' to 'ROPD'")
print("✅ Removed order book endpoint (requires subscription)")
print("✅ Added turnovers endpoint for volume data")
print("✅ Added open positions endpoint with expiration date parameter")
print("✅ Updated API references with working URLs")

print("\n" + "="*80)
print("🔗 FINAL WORKING ENDPOINTS:")
print("="*80)

print("""
# Complete Sberbank options workflow
def get_sberbank_options_complete():
    base_url = "https://iss.moex.com/iss"
    
    # 1. Get underlying assets
    assets_response = requests.get(f"{base_url}/statistics/engines/futures/markets/options/assets.json", 
                                 params={'asset_type': 'S'})
    
    # 2. Get Sberbank expirations  
    exp_response = requests.get(f"{base_url}/statistics/engines/futures/markets/options/assets/SBER.json")
    
    # 3. Get option board
    board_response = requests.get(f"{base_url}/statistics/engines/futures/markets/options/assets/SBER/optionboard.json")
    
    # 4. Get volume data
    vol_response = requests.get(f"{base_url}/statistics/engines/futures/markets/options/assets/SBER/turnovers.json")
    
    # 5. Get open positions (with expiration date)
    pos_response = requests.get(f"{base_url}/statistics/engines/futures/markets/options/assets/SBER/openpositions.json",
                               params={'date': '2025-06-25'})  # Use actual expiration date
    
    # 6. Get specific option details (use SECID from board_response)
    option_response = requests.get(f"{base_url}/engines/futures/markets/options/boards/ROPD/securities/[SECID].json")
    
    return {
        'assets': assets_response.json() if assets_response.status_code == 200 else None,
        'expirations': exp_response.json() if exp_response.status_code == 200 else None,
        'board': board_response.json() if board_response.status_code == 200 else None,
        'volume': vol_response.json() if vol_response.status_code == 200 else None,
        'positions': pos_response.json() if pos_response.status_code == 200 else None
    }
""")

print("\n" + "="*80)
print("📋 WORKING URLs FOR SBERBANK:")
print("="*80)
print("Assets:         https://iss.moex.com/iss/statistics/engines/futures/markets/options/assets.json?asset_type=S")
print("Expirations:    https://iss.moex.com/iss/statistics/engines/futures/markets/options/assets/SBER.json")
print("Option Board:   https://iss.moex.com/iss/statistics/engines/futures/markets/options/assets/SBER/optionboard.json")
print("Volume Data:    https://iss.moex.com/iss/statistics/engines/futures/markets/options/assets/SBER/turnovers")
print("Open Positions: https://iss.moex.com/iss/statistics/engines/futures/markets/options/assets/SBER/openpositions")
print("Option Details: https://iss.moex.com/iss/engines/futures/markets/options/boards/ROPD/securities/{SECID}.json")

print("\n" + "="*80)
print("🎯 READY TO USE:")
print("="*80)
print("The notebook now uses the correct MICEX API structure with:")
print("- Proper board (ROPD)")
print("- Volume data from turnovers endpoint") 
print("- Open positions with expiration date parameter")
print("- No subscription-required endpoints")
print("- All working API references included")


MICEX OPTIONS API - CORRECTED WORKFLOW

📋 UPDATED STEP-BY-STEP PROCESS:

1️⃣ GET UNDERLYING ASSETS:
   Endpoint: /iss/statistics/engines/futures/markets/options/assets.json
   Purpose: Find which assets have options available
   Parameters: asset_type='S' for stocks

2️⃣ GET ASSET EXPIRATIONS:
   Endpoint: /iss/statistics/engines/futures/markets/options/assets/[ASSET].json
   Purpose: Get expiration dates for specific asset
   Example: /iss/statistics/engines/futures/markets/options/assets/SBER.json

3️⃣ GET OPTION BOARD:
   Endpoint: /iss/statistics/engines/futures/markets/options/assets/[ASSET]/optionboard.json
   Purpose: Get all CALL and PUT options for the asset
   Parameters: date=[expiration_date] (optional)

4️⃣ GET DETAILED OPTION DATA:
   🔹 Security Info: /iss/engines/futures/markets/options/boards/ROPD/securities/[SECID].json
   🔹 Trades: /iss/engines/futures/markets/options/boards/ROPD/securities/[SECID]/trades.json
   🔹 Volume Data: /iss/statistics/engines/futures/markets/